# Synaptic Plasticity and Learning (CCNSS2026)

## Tutorial 1: Hebbian learning and Oja's rule

**Hebbian learning rule**:
$$\dot w = \eta\, y\,x.$$
Hebbian learning is local and biologically plausible, but is unstable — weights grow without bound.

**Oja's rule** adds a normalization term:

$$\Delta w = \eta\, y\,(x - y\,w), \qquad y = w^\top x$$

The subtracted $y^2 w$ keeps $\|w\|\to 1$ and, on average, drives $w$ to the **leading
eigenvector of the input covariance** — i.e. Oja is *online PCA* with a single, local,
Hebbian-plus-decay update. 

In this tutorial, we will verify this property of Oja's rule by:
1. a set of synthetic 2D data cloud (geometry);
2. a series of real MNIST images (the top principal component as a learned filter).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml   # only used to download the MNIST images

rng = np.random.default_rng(0)
plt.rcParams["figure.dpi"] = 110


## Part A — 2-D intuition

An anisotropic Gaussian cloud has one clear long axis. Oja should find it.

We draw $N = 4000$ samples from a **bivariate Gaussian** $\mathbf{x} \sim \mathcal{N}(\mathbf{0},\,\Sigma)$ in three steps:

1. **Isotropic samples.** Draw $\mathbf{z}_i \stackrel{\text{i.i.d.}}{\sim} \mathcal{N}(\mathbf{0}, I_2)$.

2. **Anisotropic scaling.** Multiply by the diagonal scale matrix $S = \operatorname{diag}(3,\;1)$:
$$\tilde{\mathbf{z}}_i = S\,\mathbf{z}_i,$$
so the two axes have standard deviations $\sigma_1 = 3$ and $\sigma_2 = 1$.

3. **Rotation.** Apply a $\theta = 30°$ counter-clockwise rotation:
$$\mathbf{x}_i = R\,\tilde{\mathbf{z}}_i, \qquad R = \begin{pmatrix}\cos\theta & -\sin\theta \\ \sin\theta & \cos\theta\end{pmatrix}.$$

The resulting distribution is $\mathcal{N}(\mathbf{0},\,\Sigma)$ with covariance
$$\Sigma = R\, S^2\, R^\top = R \begin{pmatrix}9 & 0 \\ 0 & 1\end{pmatrix} R^\top.$$
Its **eigenvalues are $\lambda_1 = 9$, $\lambda_2 = 1$** — unchanged by rotation — and the corresponding **eigenvectors are the columns of $R$**, i.e.\ the axes rotated by $\theta = 30°$. The top principal component therefore points in the direction $30°$ from the horizontal.


In [ ]:
theta = np.deg2rad(30)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
X = (rng.standard_normal((4000, 2)) @ np.diag([3.0, 1.0])) @ R.T

plt.figure(figsize=(4, 4))
plt.scatter(X[:, 0], X[:, 1], s=1, alpha=0.5)
plt.axis("equal")
plt.gca().spines[["top", "right", "bottom", "left"]].set_visible(False)
plt.xticks([])
plt.yticks([])
plt.savefig("demo1_oja_pca_data.png", bbox_inches="tight", transparent=True, pad_inches=0)

In [ ]:
# EXERCISE: compute the covariance matrix manually to verify the data structure and compute the top PC

X -= X.mean(0)
# YOUR CODE HERE
# compute the covariance matrix C = np.cov(X.T), find its eigenvalues/eigenvectors,
# and set true_pc to the eigenvector with the largest eigenvalue
raise NotImplementedError("compute the covariance matrix and top PC")


Next, we will implement Oja's rule as an online loop. We will track the alignment of the learned weight vector with the true top principal component.

$$w_{t} \leftarrow w_{t-1} + \eta\, y_t\,(x_t - y_t\,w_{t-1}), \qquad y_t = w_{t-1}^\top x_t$$

In [ ]:
def oja(X, lr=0.01, w0=None, ref=None, epochs=1):
    """Online Oja's rule. Returns final unit weight vector and (optional) alignment trace.

    epochs : number of passes over X (use >1 to give the online rule more updates to
             converge on higher-dimensional inputs such as image patches).
    """
    w = rng.standard_normal(X.shape[1]) if w0 is None else w0.copy()
    w /= np.linalg.norm(w)
    cosines = []
    # EXERCISE: implement Oja's rule as an online loop. Track alignment with the true top PC.
    for _ in range(epochs):
        for x in X:
            # YOUR CODE HERE
            # compute y = w @ x, update w with Oja's rule (Hebbian term + normalization term),
            # and if ref is not None append |w @ ref| / ||w|| to `cosines`
            raise NotImplementedError("implement Oja's rule")
    return w, np.array(cosines)

w2d, cos2d = oja(X, lr=0.01, ref=true_pc)
print("final |cos(w, true_pc)| =", round(cos2d[-1], 4), " |w| =", round(np.linalg.norm(w2d), 4))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].scatter(X[:, 0], X[:, 1], s=4, alpha=0.15)
for vec, c, lab in [(true_pc, "k", "true PC"), (w2d/np.linalg.norm(w2d), "r", "Oja w")]:
    ax[0].annotate("", xy=3*vec, xytext=(0, 0),
                   arrowprops=dict(color=c, width=2, headwidth=9))
    ax[0].plot([], [], color=c, lw=2, label=lab)
ax[0].set_aspect("equal")
ax[0].legend()
ax[0].set_title("Data cloud + learned axis")
ax[1].plot(cos2d)
ax[1].axhline(1, ls="--", c="gray")
ax[1].set_xlabel("update step")
ax[1].set_ylabel("|cos(w, true PC)|")
ax[1].set_title("Convergence to top PC")
ax[1].set_ylim(0, 1.05)
plt.tight_layout()


## Part B — MNIST images: Oja learns the top principal component

We now feed the neuron a **series of real images**: thousands of MNIST handwritten digits,
one $28\times28$ image at a time as the input vector $x\in\mathbb{R}^{784}$. We mean-center
the ensemble (subtract the average image) so the neuron learns the leading *direction of
variation* across digits rather than the trivial mean-brightness mode.

A single Oja neuron, updated online one image at a time, should converge to the ensemble's
**top principal component**. We check it against the PC computed **directly with NumPy**
(eigendecomposition of the covariance matrix `np.linalg.eigh(np.cov(...))`) — no sklearn PCA,
no batch solver inside the learning loop. *(Note: the top PC of natural images is a smooth,
low-frequency pattern — not an oriented Gabor. Oriented, sparse receptive fields require
ICA/sparse coding, which is exactly what Demo 2's Error-Gated Hebbian Rule delivers.)*


In [ ]:
# --- load a series of MNIST images as inputs ---------------------------------
mnist_X, _ = fetch_openml("mnist_784", version=1, return_X_y=True,
                          as_frame=False, parser="liac-arff")
P = 28                                            # MNIST images are 28 x 28
N = 4000                                          # number of images shown to the neuron
sel = rng.choice(len(mnist_X), size=N, replace=False)
imgs = mnist_X[sel].astype(np.float64) / 255.0    # scale pixels to [0, 1]
print("input images:", imgs.shape)                # (N, 784)

mean_img = imgs.mean(0)
X_img = imgs - mean_img                           # mean-center the ensemble (remove DC mode)

# EXERCISE: compute the top PC computed directly with NumPy
# YOUR CODE HERE
# compute the pixel covariance matrix C = np.cov(X_img.T) and set top_pc to the
# eigenvector with the largest eigenvalue
raise NotImplementedError("compute the top PC with NumPy")

# EXERCISE: implement Oja's rule: same online neuron, learning one image at a time
# YOUR CODE HERE
# call oja() on X_img (w0=rng.standard_normal(P * P), epochs=2, ref=top_pc) to get w_img, cos_img
raise NotImplementedError("run Oja's rule on the image ensemble")

w_img /= np.linalg.norm(w_img)
print("|cos(Oja filter, NumPy top PC)| =", round(abs(w_img @ top_pc), 4))


In [ ]:
# align sign for display (PCs are defined up to a sign)
if w_img @ top_pc < 0:
    w_img = -w_img

# Row 1: a sample of the actual images streamed into the neuron.
fig, ax = plt.subplots(1, 8, figsize=(9, 1.3))
for a, im in zip(ax, imgs[:8]):
    a.imshow(im.reshape(P, P), cmap="gray"); a.axis("off")
fig.suptitle("A few of the MNIST images shown to the neuron", y=1.15)
plt.tight_layout()
plt.show()

# Row 2: mean image, NumPy top PC, Oja-learned filter, convergence.
# Share one color scale across the two filters so they are mapped identically.
vmax = max(abs(top_pc).max(), abs(w_img).max())
vmin = -vmax
fig, ax = plt.subplots(1, 4, figsize=(11, 3.0))
ax[0].imshow(mean_img.reshape(P, P), cmap="gray")
ax[0].set_title("mean image")
ax[1].imshow(top_pc.reshape(P, P), cmap="RdBu_r", vmin=vmin, vmax=vmax)
ax[1].set_title("NumPy top PC")
ax[2].imshow(w_img.reshape(P, P), cmap="RdBu_r", vmin=vmin, vmax=vmax)
ax[2].set_title("Oja-learned filter")
for a in ax[:3]: a.axis("off")
ax[3].plot(cos_img)
ax[3].axhline(1, ls="--", c="gray")
ax[3].set_xlabel("update step (one image each)")
ax[3].set_ylabel("|cos(w, top PC)|")
ax[3].set_title("Convergence to top PC")
ax[3].set_ylim(0, 1.05)
plt.tight_layout()

## Takeaways

- **Oja = Hebb + normalization = online PCA.** A single, *local* synaptic rule extracts the
  direction of maximal variance — no batch eigensolver needed.
- Streaming MNIST images one at a time, the learned filter matches the **NumPy-computed top
  PC** (cosine ≈ 1).
- **Bridge to Lecture 2:** variance/PCA is one objective. What if the *right* objective is
  **maximizing information**? That yields richer, sparser, more biological features — and a
  *local* rule to learn them (the **Error-Gated Hebbian Rule**, Demo 2).
